## Setup

In [ ]:
import os
from dotenv import load_dotenv
import numpy as np
import pandas as pd
from datetime import datetime
from astropy.time import Time

In [ ]:
load_dotenv()
raw_data_path = os.getenv('MAG_RAW_PATH')
csv_data_path = os.getenv('MAG_CSV_PATH')

data = {}

BEGIN_DATE = "20100501_000000"
END_DATE = "20240921_235900"

## Reading CSVs

In [ ]:
date_format = "%Y%m%d_%H%M%S"
start_dt = datetime.strptime(BEGIN_DATE, date_format)
end_dt = datetime.strptime(END_DATE, date_format)

df_raw_list = []
current_start = start_dt

In [ ]:
def parse_tai_to_utc(tai_series: pd.Series) -> pd.Series:
    iso_strings = tai_series.str.replace('.', '-', regex=False).str.replace('_', 'T', regex=False)

    t_tai = Time(iso_strings.tolist(), format='isot', scale='tai')
    return pd.to_datetime(t_tai.utc.isot)

In [ ]:
while current_start < end_dt:
    current_end = current_start + pd.DateOffset(months=1)
    if current_end > end_dt:
        current_end = end_dt

    start_year = str(current_start.year)
    end_year = str(current_end.year)
    year_dir = os.path.join(raw_data_path, start_year)

    start_str = current_start.strftime("%Y%m%d_%H%M%S")
    end_str = current_end.strftime("%Y%m%d_%H%M%S")

    file_name = f"jsoc_data_{start_str}_TAI_to_{end_str}_TAI.csv"
    full_path = os.path.join(year_dir, file_name)

    if os.path.exists(full_path):
        df = pd.read_csv(full_path)
        df['QUALITY'] = df['QUALITY'].apply(lambda x: int(x, 16) if isinstance(x, str) else x)

        df['REGION_ID'] = df['DATASET_QUERY'].str.extract(r'\[(\d+)\]')
        df = df.drop(columns=['CONTENT', 'DATASET_QUERY'], errors='ignore')

        df['T_REC'] = df['T_REC'].str.replace('_TAI', '', regex=False)
        df['T_REC'] = parse_tai_to_utc(df['T_REC'])

        df = df.rename(columns={'T_REC': 'ds'}).set_index('ds').tz_localize('UTC')

        df_raw_list.append(df)
        print(f"Lido: {file_name}")
    else:
        print(f"Aviso: Arquivo não encontrado - {full_path}")

    current_start = current_end

print("Leitura e Agregação concluídas.")

In [ ]:
df_mag_global_raw = pd.concat(df_raw_list).sort_index()

df_mag_global_raw.index = df_mag_global_raw.index.round('12min')

df_mag_global_raw = df_mag_global_raw.reset_index()
df_mag_global_raw = df_mag_global_raw.drop_duplicates(subset=['ds', 'REGION_ID'], keep='first')
df_mag_global_raw = df_mag_global_raw.drop(columns=['REGION_ID'])
df_mag_global_raw = df_mag_global_raw.set_index('ds')

target_cols = ['USFLUX', 'R_VALUE']
cols_presentes = [c for c in target_cols if c in df_mag_global_raw.columns]
df_mag_global_raw[cols_presentes] = df_mag_global_raw[cols_presentes].replace([0, 0.0, -99999, -99999.0], np.nan)

agg_dict = {
    'USFLUX':['min', 'max', 'mean', 'sum'],
    'R_VALUE': ['min', 'max', 'mean', 'sum', 'count']
}

print("Agregando base global...")
df_mag_global = df_mag_global_raw.groupby(level='ds').agg(agg_dict)
df_mag_global.columns = [f"{col[0]}_{col[1].upper()}" for col in df_mag_global.columns]
df_mag_global = df_mag_global.rename(columns={'R_VALUE_COUNT': 'COUNT'})

df_mag_global.loc[df_mag_global['COUNT'] == 0, 'USFLUX_SUM'] = np.nan
df_mag_global.loc[df_mag_global['COUNT'] == 0, 'R_VALUE_SUM'] = np.nan

df_mag_global = df_mag_global.asfreq('12min')

In [ ]:
df_mag_global